# VoxCPM2 x Sheng (Swahili + English) - LoRA on Colab

Fine-tunes **`openbmb/VoxCPM2`** to speak **Sheng** (Kenyan Swahili/English code-switching)
from the HF dataset **`lyle-mlengineer/kenyan-celebs`** (42 clips, speaker `abel`, ~21 min).

All heavy state (16 kHz wavs, manifests, base weights, checkpoints, logs) lives on
**Google Drive**, not on the VM. That is why a disconnect is cheap:

> **After a disconnection, run cells 1-3 and then `sheng-voxcpm all --resume`.**
> Finished stages are skipped, and training resumes from
> `<artifacts>/checkpoints/lora_r32/latest`.

| Stage | Command | Notes |
|---|---|---|
| setup | `sheng-voxcpm env` | clones OpenBMB/VoxCPM, `--no-deps` + curated deps (never replaces Colab's torch) |
| data | `sheng-voxcpm data` | dataset -> 16 kHz PCM16 + train/val JSONL + `data_report.md` |
| model | `sheng-voxcpm model` | `openbmb/VoxCPM2` snapshot (~4-8 GB) |
| train | `sheng-voxcpm train` | LoRA r=32 on LM+DiT, ~20 GB VRAM |
| eval | `sheng-voxcpm eval --ab` | fixed Sheng prompts, LoRA vs base |
| export | `sheng-voxcpm export` | portable adapter + sample wavs |

GPU: **L4 (24 GB)** is the cheap choice, **A100** is faster. T4 works only with
`--profile t4 --allow-small-gpu` (no native bf16 => slow).

In [ ]:
# 1. Check the runtime hardware.
#    Runtime > Change runtime type > Hardware accelerator > GPU (L4 / A100).
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
try:
    import torch
    print("torch", torch.__version__, "| cuda", torch.version.cuda,
          "| device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
except Exception as exc:
    print("torch not importable yet:", exc)

In [ ]:
# 2. Mount Google Drive and point the pipeline at it.
#    Everything expensive is written under ARTIFACTS, which survives the VM.
from pathlib import Path

ARTIFACTS = "/content/drive/MyDrive/voxcpm2-sheng"

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("not on Colab / Drive unavailable:", exc)

if Path("/content/drive/MyDrive").is_dir():
    Path(ARTIFACTS).mkdir(parents=True, exist_ok=True)
    print("artifacts:", ARTIFACTS)
else:
    ARTIFACTS = "/content/artifacts"          # ephemeral: lost on disconnect
    print("WARNING: no Drive, falling back to", ARTIFACTS)

import os
os.environ["SHENG_ARTIFACTS"] = ARTIFACTS

In [ ]:
# 3. Get the code and install it (light deps only; torch stays as Colab shipped it).
REPO_URL = "https://github.com/lyle-mlengineer/audio-speech-ai.git"
DEST = Path("/content/audio-speech-ai")

if DEST.exists():
    !git -C {DEST} pull --ff-only
else:
    !git clone --depth 1 {REPO_URL} {DEST}

%cd {DEST}/finetune-voxcpm
!pip install -q -e .

In [ ]:
# 4. Doctor: GPU, bf16, Drive, free space, deps, base model, manifests.
#    Fix whatever FAILs (env / data / model) before training.
!sheng-voxcpm doctor

In [ ]:
# 5. One-shot pipeline. Re-run this exact cell after a disconnect.
#
#    --max-hours stops training gracefully before the Colab session limit
#    (the trainer checkpoints on SIGINT), so nothing is lost.
#    Skip eval/export on the first pass if you only care about the adapter:
#    add --skip-eval --skip-export.
!sheng-voxcpm all --resume --max-hours 8

In [ ]:
# 5b. Or drive the stages yourself (same state, same resume behaviour).
# !sheng-voxcpm env
# !sheng-voxcpm data
# !sheng-voxcpm model
# !sheng-voxcpm train --max-hours 8
# !sheng-voxcpm eval --ab
# !sheng-voxcpm export

In [ ]:
# 6. Where are we? This is the state that survived the disconnect.
!sheng-voxcpm status

In [ ]:
# 7. Listen: the five eval prompts, LoRA vs base, for each evaluated step.
from IPython.display import Audio, display

evals = sorted(Path(ARTIFACTS, "evals", "lora_r32").glob("step_*/*.wav"))
print(f"{len(evals)} clips")
for wav in evals[-12:]:
    print(wav.relative_to(ARTIFACTS))
    display(Audio(str(wav)))

In [ ]:
# 8. Package the adapter and download it (plus the eval samples).
import shutil

!sheng-voxcpm export
exports = sorted(Path(ARTIFACTS, "exports", "lora_r32").glob("step_*"))
if exports:
    bundle = shutil.make_archive("/content/sheng_voxcpm2_lora", "zip", exports[-1])
    print("bundle:", bundle)
    from google.colab import files
    files.download(bundle)
else:
    print("nothing exported yet - train first")

## Troubleshooting

| Symptom | Fix |
|---|---|
| `No CUDA GPU` | Runtime > Change runtime type > GPU, then re-run cell 1 |
| `GPU is not ideal ... VRAM < 20 GB` | You are on a T4: `--profile t4 --allow-small-gpu`, expect it to be slow |
| CUDA OOM during training | next profile down: `!sheng-voxcpm train --profile t4 --allow-small-gpu`, or `--set max_batch_tokens=4096` |
| `sample_rate mismatch` | the rendered YAML was edited by hand; re-run `sheng-voxcpm train` to regenerate `<artifacts>/configs/lora_r32.yaml` |
| `ModuleNotFoundError` in the trainer | `!sheng-voxcpm env --reinstall` (reinstalls the curated deps) |
| Session died mid-training | re-run cells 1-3, then `!sheng-voxcpm all --resume` |
| Sounds like training audio only | overfitting: evaluate an earlier step, e.g. `!sheng-voxcpm eval --steps 200 400` |

`progress.md` in the repo and `<artifacts>/progress/progress.md` hold the live journal;
machine state is `<artifacts>/progress/state.json` and `runs.jsonl`.